# 14. 그늘막·쉼터 우선지역 (과제 4, 예비) — 성남

**곤란구간(피할 수 없는 뜨거운 보행 길목) × HSI(장소 열스트레스)** → 폭염 대응시설 우선지역.

- 우선도 = 보행 매개중심성(통행 대리) × 장소 HSI. 뜨겁고 사람 많이 지나는 길목 우선.
- ⚠️ 완성판은 혼잡(06~09, COMPAS) 필요. 여기선 네트워크 중심성=통행 대리(예비).

산출물: `data/results/facility_priority.geojson`, `output/figures/fig16_facility.png`

In [1]:
# ── 입력 존재 가드 ─────────────────────────────────────────────
# 이 노트북은 08 경로분석 의 산출물을 입력으로 씁니다.
# 해당 단계가 환경 제약으로 건너뛰어졌고 동봉 산출물도 없으면 아래 셀을 건너뜁니다.
import pathlib
_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
LOCAL_RUN = (_ROOT / "data/results/bottleneck_walk.geojson").exists()
print("[실행]" if LOCAL_RUN else "[건너뜀]", "data/results/bottleneck_walk.geojson", "-", "입력 있음" if LOCAL_RUN else "입력 없음 (08 경로분석 산출물 필요)")


[실행] data/results/bottleneck_walk.geojson - 입력 있음


In [2]:
if LOCAL_RUN:
    import sys, pathlib
    sys.path.append(str(pathlib.Path.cwd().parent))
    import numpy as np, geopandas as gpd
    from src.config import DATA_RESULTS, OUT_FIGURES, CRS_LANDSAT, CRS_SUBMIT

    bn = gpd.read_file(DATA_RESULTS / "bottleneck_walk.geojson")            # 곤란구간(보행 엣지)
    hsi = gpd.read_file(DATA_RESULTS / "heat_index_final.geojson")[["hsi", "geometry"]]

    # 엣지 중점 → HSI 최근접 부착
    mid = gpd.GeoDataFrame({"eid": range(len(bn))},
                           geometry=bn.geometry.to_crs(CRS_LANDSAT).interpolate(0.5, normalized=True).values, crs=CRS_LANDSAT)
    j = gpd.sjoin_nearest(mid, hsi.to_crs(CRS_LANDSAT), how="left").drop_duplicates("eid").sort_values("eid").reset_index(drop=True)
    bn["hsi"] = j["hsi"].values
    bn["priority"] = (bn["bottleneck"] / bn["bottleneck"].max()) * bn["hsi"].fillna(bn["hsi"].median())
    bn["priority"] = bn["priority"] / bn["priority"].max()
    print("우선도 상위 (뜨겁고 피할수없는 길목):")
    print(bn.nlargest(6, "priority")[["bottleneck", "hsi", "priority"]].round(3).to_string(index=False))
    print(f"\n최우선 구간(우선도>0.6): {int((bn.priority>0.6).sum())}개 링크 = 그늘막·쿨링포그 1차 후보")

우선도 상위 (뜨겁고 피할수없는 길목):
 bottleneck   hsi  priority
      0.996 0.893     1.000
      0.996 0.893     1.000
      1.000 0.880     0.989
      0.943 0.927     0.982
      0.948 0.915     0.975
      0.924 0.927     0.963

최우선 구간(우선도>0.6): 73개 링크 = 그늘막·쿨링포그 1차 후보


In [3]:
if LOCAL_RUN:
    bn[["bottleneck", "hsi", "priority", "geometry"]].to_file(DATA_RESULTS / "facility_priority.geojson", driver="GeoJSON")

    import matplotlib.pyplot as plt
    from src.viz.fonts import setup_font
    setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
    fig, ax = plt.subplots(figsize=(11, 10))
    bn.plot(ax=ax, color="#eeeeee", linewidth=0.5)
    bn.sort_values("priority").plot(ax=ax, column="priority", cmap="YlOrRd", linewidth=2, legend=True, legend_kwds={"shrink": 0.5})
    top = bn.nlargest(40, "priority")
    top.geometry.centroid.plot(ax=ax, color="blue", markersize=25, marker="v", label="쉼터 1차 후보(상위40)")
    ax.legend(); ax.set_title("그늘막·무더위쉼터 우선지역 (곤란구간 × HSI)"); ax.set_axis_off()
    plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig16_facility.png", dpi=120); plt.close()
    print("저장 완료.")

<프로젝트>\AppData\Local\Temp\ipykernel_11792\2672905983.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  top.geometry.centroid.plot(ax=ax, color="blue", markersize=25, marker="v", label="쉼터 1차 후보(상위40)")


저장 완료.
